# Sprint 7D - Graph C GATv2 Mechanism Ablation Runner

**Runner-only notebook.** Model, graph loading, training, evaluation, plotting, and reporting logic stays in the repository under `src/`, `scripts/`, and `configs/`.

Execution plan: `docs/exec-plans/active/007d-sprint7d-graphc-gatv2-mechanism-ablation.md`  
Runner boundary: `colab/README.md`

Before starting, confirm that the approved code revision is pushed and Drive contains the raw data plus Sprint 3/Sprint 5B processed graph artifacts. If Sprint 5B Graph C S5F2 is missing, this runner rebuilds it from Sprint 3 artifacts and raw data.

## Step 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Clone Or Update Repo Checkout

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/YasinEkici/crispr-gnn-offtarget.git"
REPO_DIR="/content/crispr-gnn-offtarget"
GIT_REF="sprint7/gat-gatv2"
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR"
  git fetch origin "$GIT_REF"
  git checkout "$GIT_REF"
  git pull --ff-only origin "$GIT_REF"
else
  git clone --branch "$GIT_REF" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi
git rev-parse --short HEAD


## Step 3 - Dependency Sync And Runtime Check

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync
uv run python - <<'PY'
import torch
import torch_geometric
print('torch', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY


## Step 4 - Copy Drive Data And Processed Graph Artifacts

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found. Checked:" >&2
  printf '  %s\n' "${DRIVE_ROOT_CANDIDATES[@]}" >&2
  echo "Available MyDrive directories:" >&2
  find /content/drive/MyDrive -maxdepth 1 -type d | sort >&2
  exit 1
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
mkdir -p data/raw data/processed
if [ -d "$DRIVE_ROOT/data/raw" ]; then
  rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
else
  echo "Missing $DRIVE_ROOT/data/raw; raw data is required if Sprint 5B Graph C must be rebuilt" >&2
  exit 1
fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then
  rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
else
  echo "Missing $DRIVE_ROOT/data/processed; Sprint 3/Sprint 5B graph artifacts are required" >&2
  exit 1
fi
if [ ! -d data/processed/graphs/sprint3/graph_c_context_observation ]; then
  echo "Missing local Sprint 3 Graph C artifacts after copy: data/processed/graphs/sprint3/graph_c_context_observation" >&2
  exit 1
fi
find data/processed/graphs -maxdepth 3 -type f -name 'manifest.json' | sort


## Step 5 - Build Or Validate Sprint 5B Graph C S5F2 Artifact

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ] || [ ! -f data/processed/graphs/sprint5b/graph_c_context_observation/features_S5F2_energy.parquet ]; then
  echo "Building Sprint 5B Graph C S5F2 artifact required by Sprint 7D..."
  uv run python scripts/build_sprint5b_graph_c_energy_features.py \
    --data-config configs/data/mak2022.yaml \
    --schema-config configs/sweeps/graph_schema_ablation.yaml \
    --source-artifact-dir data/processed/graphs/sprint3 \
    --artifact-dir data/processed/graphs/sprint5b \
    --report-path outputs/sprint5b/graph_c_energy_sensitivity_artifact_report.md
fi
PYTHONPATH=src uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_C
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader
materialized = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5b')).load(GRAPH_C)
manifest = materialized.manifest
feature_tables = manifest.get('feature_tables', {})
if int(feature_tables.get('S5F2_energy', 0)) != 268:
    raise SystemExit(f'Missing 268-column Graph C S5F2_energy feature table: {feature_tables}')
if 'target_observation_features' not in feature_tables:
    raise SystemExit(f'Missing target_observation_features: {feature_tables}')
print('graph_name:', manifest.get('graph_name'))
print('split_id:', manifest.get('split_id'))
print('label_scheme:', manifest.get('label_scheme'))
print('feature_tables:', feature_tables)
PY


## Step 6 - Run Sprint 7D Mechanism Ablation

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint7d_graphc_gatv2_mechanism_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint7d_graphc_gatv2_mechanism_ablation.py \
  --config configs/sweeps/sprint7d_graphc_gatv2_mechanism_ablation.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint7d_graphc_gatv2_mechanism_run_id.txt


## Step 7 - Return Outputs To Drive

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
test -n "$DRIVE_ROOT"
RUN_BASENAME=$(cat /content/sprint7d_graphc_gatv2_mechanism_run_id.txt)
LOCAL_OUT="outputs/sprint7d"
RETURN_ROOT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
if [ -e "$RETURN_ROOT" ]; then
  echo "Output already exists in Drive: $RETURN_ROOT" >&2
  exit 1
fi
mkdir -p "$RETURN_ROOT"
rsync -a "$LOCAL_OUT/" "$RETURN_ROOT/"
find "$RETURN_ROOT" -maxdepth 3 -type f | sort | head -100


## Step 8 - Validate Output Contract

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
OUT="outputs/sprint7d"
test -f "$OUT/graphc_gatv2_mechanism_ablation.csv"
test -f "$OUT/graphc_gatv2_mechanism_ablation_report.md"
test -f "$OUT/graphc_gatv2_mechanism_ablation_run_manifest.json"
test -f "$OUT/graph_artifact_provenance.json"
test -d "$OUT/diagnostics"
test -d "$OUT/figures"
test -f "$OUT/diagnostics/graphc_gatv2_component_ablation_audit.csv"
test -f "$OUT/diagnostics/graphc_gatv2_mechanism_attention_summary.csv"
PYTHONPATH=src uv run python - <<'PY'
import json
from pathlib import Path
manifest = json.loads(Path('outputs/sprint7d/graphc_gatv2_mechanism_ablation_run_manifest.json').read_text())
ids = {run['predeclared_id'] for run in manifest['runs']}
expected = {
    'S7D_REF_XGB_F4',
    'S7D_REF_GRAPH_A_GCN',
    'S7D_REF_GRAPH_A_GATV2',
    'S7D_REF_GRAPH_C_GCN',
    'S7D_REF_FULL_GRAPH_C_GATV2',
    'S7D_R1_no_context_edges',
    'S7D_R2_edge_blind_attention',
    'S7D_R3_mask_target_context_features',
}
if ids != expected:
    raise SystemExit(f'Unexpected Sprint 7D run IDs: {ids}')
print('validated', manifest['batch_id'])
PY
